# 2. Floor-adjusted + EIP-8368

Reproduces the report's floor selection, rate-specific data multipliers, CPSB matching, central simulation, and **Floor and CPSB attribution** appendix. The common sensitivity runs in this notebook also reproduce the baseline and EIP-8368 state-demand caps and all four complete elasticity vectors.

The floor/data multiplier transports measured historical transaction-floor exposure into an aggregate demand curve. It does not estimate the extra execution or state participation response to changed BAL-related transaction bills.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/shared_fee/replay.py").is_file())
for directory in (ROOT / "src", ROOT / "scripts", ROOT / "scripts/shared_fee"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from publication_workflow import (
    DATA, REPORT, LABELS, baseline_anchor, read_table, require_files,
    run_stage, shared_display, source_snapshot, verify_sources,
)
DATA.mkdir(parents=True, exist_ok=True)
(ROOT / "plots").mkdir(exist_ok=True)
REUSE = os.environ.get("ONE_DIMENSIONAL_REUSE_OUTPUTS", "0") == "1"
REFRESH_XATU = os.environ.get("ONE_DIMENSIONAL_REFRESH_XATU", "0") == "1"
before = source_snapshot()
pd.set_option("display.max_columns", 24)
print("Repository:", ROOT)
print("Reuse generated outputs:", REUSE, "| Refresh Xatu inputs:", REFRESH_XATU)

## Derive the floor, common limit, and CPSB

The transfer construction in the report uses 21,000 gas and 221 physical bytes per transaction. Its density caps the benefit of raising the floor. For each propagation allocation, we derive the largest admissible common limit and choose the lowest sufficient calibrated floor rate. CPSB scales with the common limit to preserve the intended state-byte target.

$$
L_{\mathrm{shared}}=\min\!\left[L_{\mathrm{execution}},(21000/221)B_{\max}\right],
\quad F^*=\left\lceil L_{\mathrm{shared}}/B_{\max}\right\rceil,
\quad \mathrm{CPSB}=1530\,L_{\mathrm{shared}}/(150\mathrm{M}).
$$

The actual candidate grid uses rates 40, 50, and every integer from 64 through 96. Each candidate selects the lowest sufficient rate from that calibrated grid.

In [ ]:
from shared_fee.optimization import (
    physical_capacities, minimum_sufficient_floor_rate, cpsb_for_limit,
    CALIBRATED_FLOOR_RATES,
)
schedule = []
for t in [3., 3.5, 4., 4.5, 5.]:
    physical = physical_capacities(t)
    limit = physical["maximum_candidate_limit"]
    rate = minimum_sufficient_floor_rate(limit, physical["safe_payload_bytes"])
    schedule.append(dict(propagation_time_s=t, shared_limit=limit, floor_rate=rate,
                         cpsb=cpsb_for_limit(limit, True), **physical))
schedule = pd.DataFrame(schedule)
assert schedule.floor_rate.tolist() == [96, 82, 64, 50, 40]
display(schedule[["propagation_time_s", "shared_limit", "floor_rate", "cpsb",
                  "physical_binding_constraint"]])

## Recalibrate data metering at each floor rate

This executes the transaction-level floor calculation across the full rate grid, with 400 bootstrap replications. The 96-gas setting supplies the detailed activation/coverage appendix; the baseline continues to use its own 64-gas row. Notebook 1 must have built the transaction panel first.

In [ ]:
run_stage("sweep_floor_rates.py",
          "--rates", *map(str, CALIBRATED_FLOOR_RATES),
          "--central-rate", "96", "--bootstrap", "400",
          outputs=["floor_rate_multiplier_sweep.csv", "equilibrium_anchor_by_floor_rate.csv",
                   "floor_rate_activation_by_class.csv", "floor_rate_multiplier_bootstrap.csv",
                   "state_creating_tx_runtime_component_coverage.csv",
                   "state_creating_tx_bal_floor_coverage.csv"], reuse=REUSE)
curve = read_table("floor_rate_multiplier_sweep.csv")
anchors = read_table("equilibrium_anchor_by_floor_rate.csv")
assert set(curve.floor_rate) == set(CALIBRATED_FLOOR_RATES)
schedule = schedule.merge(anchors[["floor_rate", "m_data"]], on="floor_rate", validate="one_to_one")
state_anchor = anchors.loc[anchors.floor_rate.eq(64), "m_state"].iloc[0]
schedule["m_state"] = state_anchor * schedule.cpsb / 1530
display(schedule[["propagation_time_s", "shared_limit", "floor_rate", "cpsb", "m_state", "m_data"]])
display(curve[["floor_rate", "m_data_block", "m_data_block_p05", "m_data_block_p95",
               "bal_affected_tx_share"]])
np.testing.assert_allclose(schedule.m_data, [4.0325, 3.0875, 2.1615, 1.7634, 1.5887],
                           atol=0.00005, rtol=0)
display(read_table("floor_rate_activation_by_class.csv"))
display(read_table("state_creating_tx_runtime_component_coverage.csv"))
display(read_table("state_creating_tx_bal_floor_coverage.csv"))
for suffix in ["static_content_decile", "state_creating_status", "contract_deployment",
               "transaction_type"]:
    display(read_table(f"eip8279_floor_activation_by_{suffix}.csv"))

## Common-limit sweep and floor/CPSB attribution

The existing runner evaluates all 380 combinations, including baseline, CPSB-only adjustment, floor-only adjustment, and both adjustments. Only payload-feasible configurations enter selection. The selected row maximizes mean delivered execution at each propagation allocation; parent execution and metered execution have the same ranking because the execution multiplier is constant.

Keeping the full grid reproduces the attribution appendix and provides central cases for the later elasticity experiment.

In [ ]:
run_stage("run_optimized_factorial.py", outputs=[
    "shared_fee_factorial_scenarios.csv", "shared_fee_factorial_paths.csv",
    "shared_fee_factorial_designs.csv", "shared_fee_factorial_manifest.csv",
], reuse=REUSE)
factorial = read_table("shared_fee_factorial_scenarios.csv")
paths = read_table("shared_fee_factorial_paths.csv")
assert len(factorial) == 380 and len(paths) == 380 * 32
feasible = factorial[factorial.floor_payload_feasible]
selected = feasible.loc[feasible.groupby(["benchmark", "propagation_time_s"])
                        .included_execution_metered.idxmax()].sort_values(["benchmark", "propagation_time_s"])
adjusted = selected[selected.benchmark.eq("fully_optimized")]
display(shared_display(adjusted).round(4))
display(selected[["benchmark", "propagation_time_s", "shared_limit", "floor_rate", "cpsb",
                  "equilibrium_fee_wei", "included_execution", "included_execution_metered",
                  "annualized_state_growth_gib", "shared_limit_hit_fraction"]])
np.testing.assert_allclose(adjusted.included_execution_metered / 1e6,
                           [70.0, 70.2, 69.4, 68.5, 67.5], atol=0.05, rtol=0)
import make_figures
make_figures.floor_activation()
make_figures.optimized_factorial()
display(Image(filename=str(ROOT / "plots/shared_fee_8279_floor_activation.png")))
display(Image(filename=str(ROOT / "plots/shared_fee_optimized_factorial.png")))

## State-demand saturation: baseline and EIP-8368

Re-solve and replay the two central configurations at every propagation allocation with unrestricted demand and caps of 1.5×, 2×, 3×, 4×, and 5× the state anchor. Caps apply to price-driven expansion before multiplying the original state shock. These are the report's full 60-case state-tail results.

In [ ]:
run_stage("run_optimized_state_tail.py", outputs=[
    "shared_fee_state_tail.csv", "shared_fee_state_tail_paths.csv",
    "shared_fee_proposal_state_tail.csv", "shared_fee_optimized_state_tail.csv",
], reuse=REUSE)
tail = read_table("shared_fee_state_tail.csv")
assert len(tail) == 60
display(tail[["benchmark", "propagation_time_s", "state_demand_cap_label",
              "equilibrium_fee_wei", "equilibrium_binding_branch",
              "included_execution_metered", "annualized_state_growth_gib",
              "shared_limit_hit_fraction", "regular_binding_fraction"]])
make_figures.optimized_state_tail()
display(Image(filename=str(ROOT / "plots/shared_fee_proposal_state_tail.png")))
display(Image(filename=str(ROOT / "plots/shared_fee_optimized_state_tail.png")))

run_stage("build_optimized_comparison.py",
          outputs=["shared_fee_optimized_comparison.csv"], reuse=REUSE)

## Full-vector elasticity sensitivity and candidate selection

These common runs cover baseline and EIP-8368 together and prepare the EIP-7999 selections needed by Notebook 3. First replay the 40 largest-limit shared-fee settings and available EIP-7999 selections. Then evaluate 708 feasible common-limit/vector settings (177 central cases reused, 531 alternative-vector cases), preserving the recovered workload's original 35-day price adjustment.

The robustness runner resumes a complete cached surface when available. On a fresh reconstruction it computes that surface. Frozen central configurations are distinct from designs reselected under each vector. Missing eligible EIP-7999 configurations remain explicitly unavailable.

In [ ]:
require_files([ROOT / "data/7999/slot_time_parameter_surface_one_at_a_time.csv",
               ROOT / "data/7999/slot_time_historical_benchmark_frontier.csv"])
run_stage("run_elasticity_comparison.py", outputs=[
    "shared_fee_elasticity_comparison.csv", "shared_fee_elasticity_comparison_paths.csv",
    "shared_fee_elasticity_comparison_manifest.json",
], reuse=REUSE)
run_stage("run_elasticity_robustness.py", outputs=[
    "shared_fee_elasticity_surface.csv", "shared_fee_elasticity_surface_paths.csv",
    "shared_fee_elasticity_reselected.csv", "shared_fee_elasticity_best_designs.csv",
    "shared_fee_elasticity_fixed_central.csv", "shared_fee_elasticity_gains.csv",
    "shared_fee_elasticity_gain_paths.csv", "shared_fee_elasticity_robustness_manifest.json",
], reuse=REUSE)
elasticity = read_table("shared_fee_elasticity_surface.csv")
assert len(elasticity) == 708
assert len(read_table("shared_fee_elasticity_surface_paths.csv")) == 22_656
fixed = read_table("shared_fee_elasticity_fixed_central.csv")
display(fixed[fixed.benchmark.isin(["proposal_faithful", "fully_optimized"])][[
    "benchmark", "window_days", "propagation_time_s", "configuration",
    "metered_execution_gas", "annualized_state_growth_gib", "hard_limit_fraction",
]])
verify_sources(before)